# Unified WSSS: DINOv3 + SAM Distillation

This notebook demonstrates a clean implementation of an online self-reinforcing knowledge distillation loop for Weakly-Supervised Semantic Segmentation (WSSS). 
We train a single model that finds objects using weak image-level tags, automatically queries SAM, and learns to produce dense pseudo-masks purely from DINO representation.

In [1]:
import torch
import torchvision
from torchvision.transforms import v2
from torch.utils.data import DataLoader
from segment_anything import sam_model_registry, SamPredictor

# 1. Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Load DINOv3 (Frozen Foundation Model)
print("Loading DINOv3 (ViT-Large)...")
dino_repo_dir = "./dinov3"
dino_model = torch.hub.load(
    dino_repo_dir,
    "dinov3_vitl16",
    source="local",
    weights="dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth",
).to(device)
dino_model.eval()

# 3. Load SAM (Segment Anything Model)
print("Loading SAM (vit_b)...")
sam_checkpoint = "./sam_vit_b_01ec64.pth"
sam = sam_model_registry["vit_b"](checkpoint=sam_checkpoint)
sam.to(device=device)
predictor = SamPredictor(sam)
print("Models loaded successfully.")

Using device: cuda
Loading DINOv3 (ViT-Large)...
Loading SAM (vit_b)...
Models loaded successfully.


### Data Loading (PASCAL VOC)

We load PASCAL VOC and convert pixel-level annotations into naive image-level vectors representing mere "tags" (which classes are present somewhere in the image).

In [2]:
import numpy as np
import os

VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 
    'car', 'cat', 'chair', 'cow', 'diningtable', 'dog', 
    'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 
    'sofa', 'train', 'tvmonitor'
]

def make_transform(resize_size: int = 448):
    to_tensor = v2.ToImage()
    resize = v2.Resize((resize_size, resize_size), antialias=True)
    to_float = v2.ToDtype(torch.float32, scale=True)
    normalize = v2.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    )
    return v2.Compose([to_tensor, resize, to_float, normalize])

transform = make_transform()

# Wrap VOC to return tracking indices. This allows us to safely cache DINO features 
# directly in memory and fetch them by dataset index later!
class WSSSDataset(torch.utils.data.Dataset):
    def __init__(self, voc_dataset, transform):
        self.voc = voc_dataset
        self.transform = transform
        
    def __len__(self): return len(self.voc)
    
    def __getitem__(self, idx):
        img, mask = self.voc[idx]
        img_t = self.transform(img)
        
        mask_t = torch.tensor(np.array(mask), dtype=torch.long)
        unique_classes = torch.unique(mask_t)
        unique_classes = unique_classes[(unique_classes != 0) & (unique_classes != 255)]
        
        label_vec = torch.zeros(20, dtype=torch.float32)
        if len(unique_classes) > 0:
            label_vec[unique_classes - 1] = 1.0
            
        return idx, img_t, label_vec

def wsss_collate_fn(batch):
    indices = torch.tensor([b[0] for b in batch], dtype=torch.long)
    images = torch.stack([b[1] for b in batch])
    labels = torch.stack([b[2] for b in batch])
    return indices, images, labels

# Download VOC 2012
voc_raw = torchvision.datasets.VOCSegmentation(
    root="./data", 
    year="2012", 
    image_set="train", 
    download=False
)
voc_segmentation = WSSSDataset(voc_raw, transform)

batch_size = 16
num_workers = min(4, os.cpu_count() or 1)

# We define TWO dataloaders. One sequential (for caching), one shuffled (for training).
seq_dataloader = DataLoader(voc_segmentation, batch_size=batch_size, collate_fn=wsss_collate_fn, shuffle=False, num_workers=num_workers, pin_memory=True)
dataloader = DataLoader(voc_segmentation, batch_size=batch_size, collate_fn=wsss_collate_fn, shuffle=True, num_workers=num_workers, pin_memory=True)

print(f"Dataset contains {len(voc_segmentation)} images configued across {num_workers} parallel workers.")

Dataset contains 1464 images configued across 4 parallel workers.


### Unified WSSS Segmenter Architecture

This uses the spatial representation generated by DINO, processes it through multiple upconv networks mapping to a dense high-res 224x224 predicted segmentation output. It handles both generating the rough "where" mapping to prompt SAM, and learning SAM's boundaries directly.

In [3]:
import torch.nn as nn
import torch.nn.functional as F

class UnifiedWSSSSegmenter(nn.Module):
    def __init__(self, in_channels=1024, num_classes=20):
        super().__init__()
        self.in_channels = in_channels
        
        # Switched from ConvTranspose2d (causes checkerboarding!) to Bilinear Upsampling + Conv2d
        # This creates much smoother, continuous mask predictions without geometric artifacts
        self.decoder = nn.Sequential(
            nn.Conv2d(in_channels, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(512, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(128, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            
            nn.Conv2d(32, num_classes, kernel_size=1)
        )
        
    def forward(self, patch_tokens, grid_size, target_size):
        # Reshape DINO tokens: [B, 196, 1280] -> [B, 1280, 14, 14]
        # Dynamically reference self.in_channels so we can swap down DINOv3 model sizes!
        x = patch_tokens.permute(0, 2, 1).reshape(-1, self.in_channels, *grid_size)
        
        # Dense Upsampling through the safe convolutional decoder
        logits = self.decoder(x) 
        
        # Dynamically scale to match whatever the exact image dimension is (e.g. 448x448)
        dense_logits = F.interpolate(logits, size=target_size, mode='bilinear', align_corners=False)
        
        # Image-level tags (Global Average Pooling) -> forces model finding object somewhere in image
        image_level_logits = dense_logits.mean(dim=(2, 3)) 
        
        return image_level_logits, dense_logits

### The Training Loop

1. Frozen DINO extracts base representation.
2. The unified segmenter yields both multi-label categorical outputs (`BCEWithLogitsLoss` supervision) and pixel-level map predictions.
3. Class-specific CAM masks are processed for object peaks serving as zero-shot prompts to SAM directly on the GPU.
4. `Dice` loss penalizes the Dense Masks produced against SAM's generated ground truth masks.

In [4]:
from tqdm.notebook import tqdm

print("--- HUGE SPEED IMPROVEMENT 5: DINOv3 OFFLINE CACHING (IN VRAM) ---")
print("Since our image transforms (make_transform) are deterministic with no random cropping or flipping,")
print("we can completely skip running DINOv3 during epochs by saving its 1024-dimension patches directly into VRAM!")

# DINOv3 ViT-Large produces 1024 channels. 448x448 input creates a 28x28 grid = 784 patches.
num_images = len(voc_segmentation)
# We now store this tensor directly on the GPU (device) to skip CPU->GPU transfer during training!
dino_cache_tensor = torch.zeros((num_images, 784, 1024), dtype=torch.float16, device=device)

dino_model.eval()
with torch.no_grad():
    for indices, images, _ in tqdm(seq_dataloader, desc="Caching DINO Tokens to VRAM"):
        images = images.to(device, non_blocking=True)
        # Using Mixed Precision to evaluate frozen DINO faster!
        with torch.amp.autocast('cuda'):
            features = dino_model.forward_features(images)
            # Store as FP16 directly in VRAM!
            patch_tokens = features['x_norm_patchtokens'].half() 
            
        for i, idx in enumerate(indices):
            dino_cache_tensor[idx] = patch_tokens[i]

mem_usage_gb = dino_cache_tensor.element_size() * dino_cache_tensor.nelement() / (1024**3)
print(f"✅ Fast-track caching complete! DINO tensors safely stored in GPU VRAM using {mem_usage_gb:.2f} GB.")

--- HUGE SPEED IMPROVEMENT 5: DINOv3 OFFLINE CACHING (IN VRAM) ---
Since our image transforms (make_transform) are deterministic with no random cropping or flipping,
we can completely skip running DINOv3 during epochs by saving its 1024-dimension patches directly into VRAM!


Caching DINO Tokens to VRAM:   0%|          | 0/92 [00:00<?, ?it/s]

✅ Fast-track caching complete! DINO tensors safely stored in GPU VRAM using 2.19 GB.


In [ ]:
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.amp import GradScaler, autocast 

# Instantiate the decoder 
unified_segmenter = UnifiedWSSSSegmenter(num_classes=20).to(device)

def dice_loss(pred, target, smooth=1.):
    pred_sig = pred
    intersection = (pred_sig * target).sum()                            
    union = pred_sig.sum() + target.sum()                               
    return 1 - (2. * intersection + smooth) / (union + smooth)

# Set `set_to_none=True` on zero_grad
optimizer_unified = optim.Adam(unified_segmenter.parameters(), lr=1e-4)
scaler = GradScaler('cuda')
criterion_tags = nn.BCEWithLogitsLoss()

num_epochs = 10 
best_example = {'loss': float('inf')}
worst_example = {'loss': 0.0}

print("Starting Unified Distillation ...")

mean_pt = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
std_pt  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

for epoch in range(num_epochs):
    unified_segmenter.train()
    
    # Notice we unpack indices!
    for i, (indices, images, image_level_labels) in enumerate(tqdm(dataloader, desc=f"Epoch {epoch+1} Distillation")):
        images = images.to(device, non_blocking=True)
        image_level_labels = image_level_labels.to(device, non_blocking=True)
        B, C, H, W = images.shape
        
        optimizer_unified.zero_grad(set_to_none=True)
        
        # --- MIXED PRECISION FORWARD PASS ---
        with autocast('cuda'):
            # 1. FETCH CACHED DINO FEATURES O(1)
            patch_tokens = dino_cache_tensor[indices].float()
            grid_dim = int(patch_tokens.shape[1] ** 0.5) # 28
                
            # 2. FORWARD UNIFIED MODEL
            img_logits, dense_logits = unified_segmenter(patch_tokens, grid_size=(grid_dim, grid_dim), target_size=(H, W))
            loss_cls = criterion_tags(img_logits, image_level_labels)
        
        # --- STRICT SAM NO GRAD (BATCHED OFF CACHED FEATURES) ---
        with torch.no_grad():
            pseudo_labels_batch = torch.zeros((B, 20, H, W), dtype=torch.float32, device=device)
            images_255 = torch.clamp((images * std_pt) + mean_pt, 0.0, 1.0) * 255.0
            
            # --- HUGE SPEED IMPROVEMENT 6: BATCHED SAM EVALUATION ---
            # Instead of a python loop processing SAM image by image, we format a batched 
            # python list of dicts that native SAM uses to run parallel batched evaluation across the GPU!
            sam_batched_input = []
            
            for b_idx in range(B):
                active_classes = torch.where(image_level_labels[b_idx] > 0)[0]
                if len(active_classes) == 0: continue
                    
                # Resize specifically for SAM's input pipeline natively (1024x1024)
                img_tensor_sam = F.interpolate(images_255[b_idx].unsqueeze(0), size=(1024, 1024), mode='bilinear')
                
                pts, lbs, classes_to_gen = [], [], []
                
                for cls_idx in active_classes:
                    cls_idx = cls_idx.item()
                    classes_to_gen.append(cls_idx)
                    cam = dense_logits[b_idx, cls_idx].detach()
                    cam_norm = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
                    max_flat = cam_norm.argmax()
                    
                    max_y, max_x = torch.div(max_flat, W, rounding_mode='trunc'), max_flat % W
                    
                    # SAM expects coordinates scaled up to its internal tensor space!
                    scaled_x = max_x.item() * (1024 / W)
                    scaled_y = max_y.item() * (1024 / H)
                    
                    pts.append([scaled_x, scaled_y])
                    lbs.append(1)
                    
                pt_batch = torch.tensor(pts, device=device).unsqueeze(1) # Nx1x2
                lb_batch = torch.tensor(lbs, device=device).unsqueeze(1) # Nx1
                
                # SAM expects image input as (C, H, W) [and float/byte depends on standard].
                # Actually, SAM requires batched [C, H, W] images specifically in its format!
                sam_batched_input.append({
                    'image': images_255[b_idx].byte(), # SAM `.forward()` deals with byte images directly!
                    'point_coords': pt_batch,
                    'point_labels': lb_batch,
                    'original_size': (H, W),
                    'classes_to_gen': classes_to_gen,
                    'b_idx': b_idx
                })
                
            if len(sam_batched_input) > 0:
                # RUN PARALLEL BATCHED CALLS
                # SAM ViT-H needs a lot of VRAM for 1024x1024 image encoding.
                # We chunk our batched_input array so we don't OOM.
                # It is possible a chunk size of 4 still uses too much VRAM on 16GB.
                # Let's drop it to 2.
                sam_chunk_size = 2
                batched_outputs = []
                for c_idx in range(0, len(sam_batched_input), sam_chunk_size):
                    chunk = sam_batched_input[c_idx:c_idx + sam_chunk_size]
                    chunk_outputs = sam(chunk, multimask_output=False)
                    batched_outputs.extend(chunk_outputs)
                
                for output, inp in zip(batched_outputs, sam_batched_input):
                    b_idx = inp['b_idx']
                    for idx, cls_idx in enumerate(inp['classes_to_gen']):
                        pseudo_labels_batch[b_idx, cls_idx] = output['masks'][idx, 0]

        # --- LOSS & AUTO-SCALED BACKWARD ---
        with autocast('cuda'):
            loss_seg_bce = F.binary_cross_entropy_with_logits(dense_logits, pseudo_labels_batch, reduction='none')
            loss_seg_bce = (loss_seg_bce * image_level_labels.unsqueeze(-1).unsqueeze(-1)).mean()
            
            pred_probs = torch.sigmoid(dense_logits)
            loss_seg_dice = dice_loss(pred_probs * image_level_labels.unsqueeze(-1).unsqueeze(-1), pseudo_labels_batch)
            
            total_loss = loss_cls + loss_seg_bce + (0.5 * loss_seg_dice)
            
        scaler.scale(total_loss).backward()
        scaler.step(optimizer_unified)
        scaler.update()
        
        # Track Best vs Worst Examples for visualization
        if total_loss.item() > worst_example['loss']:
            worst_example = {
                'loss': total_loss.item(),
                'image': images_255[b_idx].permute(1,2,0).cpu().numpy() / 255.0,
                'sam_pseudolabel': pseudo_labels_batch[-1].cpu(),
                'prediction': pred_probs[-1].detach().cpu(),
                'active_classes': active_classes.cpu().numpy()
            }
        if total_loss.item() < best_example['loss'] and total_loss.item() > 0:
            best_example = {
                'loss': total_loss.item(),
                'image': images_255[b_idx].permute(1,2,0).cpu().numpy() / 255.0,
                'sam_pseudolabel': pseudo_labels_batch[-1].cpu(),
                'prediction': pred_probs[-1].detach().cpu(),
                'active_classes': active_classes.cpu().numpy()
            }

    print(f"Epoch [{epoch+1}/{num_epochs}] Complete | Avg Loss: {total_loss.item():.4f}")

# --- Visualize The Final Results ---
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for idx, (row_ax, ex, label) in enumerate(zip(axes, [worst_example, best_example], ['Worst-Performing Batch Image', 'Best-Performing Batch Image'])):
    if 'image' not in ex: continue
    row_ax[0].imshow(ex['image'])
    row_ax[0].set_title(f"{label}\nLoss: {ex['loss']:.3f}")
    
    cls_to_show = ex['active_classes'][0] if len(ex['active_classes']) > 0 else 0
    cls_name = VOC_CLASSES[cls_to_show] if len(ex['active_classes']) > 0 else "None"
    
    row_ax[1].imshow(ex['image'])
    row_ax[1].imshow(ex['sam_pseudolabel'][cls_to_show].numpy(), cmap='Greens', alpha=0.6)
    row_ax[1].set_title(f"SAM pseudo-GT / Class: {cls_name}")
    
    row_ax[2].imshow(ex['image'])
    row_ax[2].imshow(ex['prediction'][cls_to_show].numpy() > 0.5, cmap='Reds', alpha=0.6)
    row_ax[2].set_title(f"Unified Model Prediction")
    
for ax in axes.flatten():
    ax.axis('off')

plt.tight_layout()
plt.show()

Starting Unified Distillation ...


Epoch 1 Distillation:   0%|          | 0/92 [00:00<?, ?it/s]

Epoch [1/10] Complete | Avg Loss: 1.1594


Epoch 2 Distillation:   0%|          | 0/92 [00:00<?, ?it/s]

### Analysis of the Distillation Results

Looking at the plots above, we can observe the dynamics of our self-teaching network:
- **The Pseudo-GT (Green):** This is SAM's output. SAM has no idea what a "cow" or a "bus" is—it only knows how to segment based on the single point prompt we gave it. If our model learns a good feature representation, its peak activation will fall right on the subject, causing SAM to draw a perfect boundary.
- **The Model Prediction (Red):** This is our `UnifiedWSSSSegmenter`'s dense layer. It is trying to warp its latent DINO tokens to match SAM's boundaries via the structural representation learned by the Dice Loss.
- **Best vs. Worst:** In the "Best" example, the model's classification peak successfully hit the object, SAM made a clean mask, and the model easily fit the DINO tokens to that mask. In the "Worst" example, the model might have picked up a confusing background element, causing SAM to mask the wrong area, leading to high loss convergence.

#### Production Inference (Zero-Shot & Zero-SAM)
The most powerful aspect of this pipeline is that once training is completed, **we completely discard SAM, the ground-truth tags, and the loops!** The model now intrinsically knows both *what* the object is and *where* it is in a single forward pass.

In [ ]:
# Production Inference Example
# Simulating taking a random image and pushing it through the trained model without loops!

unified_segmenter.eval()

# Grab a random batch from the dataloader acting as validation
_, val_images, val_labels = next(iter(dataloader))
val_img = val_images[1].unsqueeze(0).to(device) # Shape: [1, 3, H, W]
B, C, H, W = val_img.shape

with torch.no_grad():
    # 1. Extract DINO tokens LIVE (since this is inference on an unknown image)
    features = dino_model.forward_features(val_img)
    patch_tokens = features['x_norm_patchtokens']
    grid_dim = int(patch_tokens.shape[1] ** 0.5)
    
    # 2. Single forward pass! NO SAM. NO IMAGE-LEVEL TAG STRIPPING.
    img_logits, dense_logits = unified_segmenter(patch_tokens, grid_size=(grid_dim, grid_dim), target_size=(H, W))
    
    # 3. Convert raw logits to probabilities (0.0 - 1.0)
    tag_probs = torch.sigmoid(img_logits[0])
    mask_probs = torch.sigmoid(dense_logits[0])
    
    # 4. Filter instantly: What classes did the model find with > 50% confidence?
    # This automatically drops classes that are not in the image.
    detected_class_indices = torch.where(tag_probs > 0.5)[0]

# --- Visualize The Detection ---
# Un-normalize image for matplotlib
img_show = val_img.squeeze().permute(1,2,0).cpu().numpy()
img_show = img_show * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
img_show = np.clip(img_show, 0, 1)

if len(detected_class_indices) == 0:
    print("The model's image-level tagger acts as a gatekeeper.")
    print("No confident objects were detected in this random sample, so no masks were evaluated. Efficient!")
else:
    print(f"Model confidently detected {len(detected_class_indices)} objects in the image without ground truth!")
    
    fig, axes = plt.subplots(1, len(detected_class_indices) + 1, figsize=(6 * (len(detected_class_indices) + 1), 6))
    
    if len(detected_class_indices) == 1:
        axes = [axes[0], axes[1]]
        
    axes[0].imshow(img_show)
    axes[0].set_title("Standard Inference Input", fontsize=14)
    axes[0].axis('off')
    
    # Iterate through ONLY the detected classes to render masks
    for i, cls_idx in enumerate(detected_class_indices):
        class_name = VOC_CLASSES[cls_idx]
        confidence = tag_probs[cls_idx].item()
        
        # Here we instantly grab the high-res mask prediction without looping!
        final_binary_mask = mask_probs[cls_idx].cpu().numpy() > 0.5
        
        axes[i+1].imshow(img_show)
        axes[i+1].imshow(final_binary_mask, cmap='Reds', alpha=0.6)
        axes[i+1].set_title(f"Predicted class: {class_name.upper()}\nConfidence: {confidence:.1%}", fontsize=14)
        axes[i+1].axis('off')
        
    plt.tight_layout()
    plt.show()